# 02 · Data Preprocessing

**Goal:** clean the raw dataset, engineer numeric features, encode categorical variables,
and prepare train/test splits for modeling.

This notebook is self-contained — it reloads `Mobile Dataset.csv` from scratch (it does not
depend on `01_eda.ipynb` having been run first). At the end, it **saves the processed splits,
encoders, and scaler to `artifacts/`** so that `03_training_and_prediction.ipynb` can load
them without repeating this work.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

## 2.0 Load the raw dataset

In [ ]:
DATA_PATH = r"Mobile Dataset.csv"
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)

## 2.1 (Optional) keep only original / synthetic / all rows

`product_name` values that look like `"brand-123"` are synthetic rows added to the dataset.
`SUBSET_MODE` controls which rows are kept:
- `"all"`
- `"original_only"`
- `"synthetic_only"`

In [ ]:
SUBSET_MODE = "synthetic_only"

df["is_synthetic_like"] = df["product_name"].str.contains(
    r"\b[a-z]+-\d+(?:-[a-z]+)?\b", regex=True
)

if SUBSET_MODE == "original_only":
    df = df[~df["is_synthetic_like"]].copy()
elif SUBSET_MODE == "synthetic_only":
    df = df[df["is_synthetic_like"]].copy()
elif SUBSET_MODE != "all":
    raise ValueError(f"Unknown SUBSET_MODE: {SUBSET_MODE!r}")

df = df.drop(columns=["is_synthetic_like"])
print(f"SUBSET_MODE = {SUBSET_MODE!r} -> {len(df):,} rows kept")

## 2.2 Clean numeric columns and drop non-predictive columns

`ram` / `storage` / `battery` have `"GB"`/`"TB"`/`"mAh"` suffixes → strip and convert to
numeric. `price` / `rating` are converted to numeric as a safety net. `category` is constant
(`"Smartphone"` for every row) so it carries no information; `product_name` and `url` are
just identifiers, not predictive features.

In [ ]:
processed_df = df.copy()

processed_df["price"]  = pd.to_numeric(processed_df["price"], errors="coerce")
processed_df["rating"] = pd.to_numeric(processed_df["rating"], errors="coerce")

processed_df["ram"] = processed_df["ram"].astype(str).str.extract(r"(\d+)").astype(float)
processed_df["storage"] = (
    processed_df["storage"].astype(str).str.replace("TB", "000", regex=False)
    .str.extract(r"(\d+)").astype(float)
)
processed_df["battery"] = processed_df["battery"].astype(str).str.extract(r"(\d+)").astype(float)

# Drop rows with missing values in key columns (none expected, kept for robustness)
processed_df = processed_df.dropna(subset=["price", "rating", "ram", "storage", "battery"])

# Drop columns with no predictive value
processed_df = processed_df.drop(columns=["category", "url", "product_name"], errors="ignore")

print("Shape after cleaning:", processed_df.shape)
processed_df.head()

## 2.3 Encode categorical columns (`brand`, `platform`)

Label encoding was chosen after comparing Label / Ordinal / Binary / One-Hot encodings
across all three models (see report for the full comparison table).

In [ ]:
encoders = {}
for col in ["brand", "platform"]:
    le = LabelEncoder()
    processed_df[col] = le.fit_transform(processed_df[col])
    encoders[col] = le   # keep each column's encoder so we can decode later

## 2.4 Feature engineering

In [ ]:
processed_df["total_memory"]    = processed_df["ram"] + processed_df["storage"]
processed_df["ram_storage"]     = processed_df["ram"] * processed_df["storage"]
processed_df["battery_per_ram"] = processed_df["battery"] / processed_df["ram"]
processed_df["storage_per_ram"] = processed_df["storage"] / processed_df["ram"]
processed_df["rating_ram"]      = processed_df["rating"] * processed_df["ram"]
processed_df["rating_storage"]  = processed_df["rating"] * processed_df["storage"]
processed_df["rating_battery"]  = processed_df["rating"] * processed_df["battery"]

print("Final feature set:", processed_df.columns.tolist())

In [ ]:
# Correlation heatmap after feature engineering
plt.figure(figsize=(10, 8))
sns.heatmap(processed_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap (After Feature Engineering)")
plt.show()

## 2.5 Train/test split + scaling

Scaling is only needed for Linear Regression; tree-based models (Random Forest, XGBoost)
work directly on unscaled features.

In [ ]:
X = processed_df.drop(columns=["price"])
y = processed_df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_test_lr  = scaler.transform(X_test)

X_train_tree = X_train.copy()
X_test_tree  = X_test.copy()

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)

## 2.6 Save artifacts for the next notebook

Everything `03_training_and_prediction.ipynb` needs — the splits, the encoders, and the
fitted scaler — is written to `artifacts/` here.

In [ ]:
joblib.dump(X_train, ARTIFACTS_DIR / "X_train.pkl")
joblib.dump(X_test, ARTIFACTS_DIR / "X_test.pkl")
joblib.dump(y_train, ARTIFACTS_DIR / "y_train.pkl")
joblib.dump(y_test, ARTIFACTS_DIR / "y_test.pkl")
joblib.dump(X_train_lr, ARTIFACTS_DIR / "X_train_lr.pkl")
joblib.dump(X_test_lr, ARTIFACTS_DIR / "X_test_lr.pkl")
joblib.dump(X_train_tree, ARTIFACTS_DIR / "X_train_tree.pkl")
joblib.dump(X_test_tree, ARTIFACTS_DIR / "X_test_tree.pkl")
joblib.dump(scaler, ARTIFACTS_DIR / "scaler.pkl")
joblib.dump(encoders, ARTIFACTS_DIR / "encoders.pkl")

print("Saved preprocessing artifacts to:", ARTIFACTS_DIR.resolve())
print(sorted(p.name for p in ARTIFACTS_DIR.glob("*.pkl")))

---
**Next:** `03_training_and_prediction.ipynb` — model training, evaluation, and the
prediction-based insights (price sensitivity, brand premium).